# Uncertain-input GP

連続入力の測定誤差とカテゴリ同定の不確かさを、output noise と区別して扱います。


## 連続入力の不確かさ


In [ ]:
from botorch.fit import fit_gpytorch_mll
import torch
from robotorchan.models import UncertainInputSingleTaskGP, UncertainCategoricalSingleTaskGP

torch.manual_seed(0)
torch.set_default_dtype(torch.double)
X = torch.linspace(0, 1, 10).unsqueeze(-1)
Y = torch.sin(2 * torch.pi * X)
X_std = torch.full_like(X, 0.03)
model = UncertainInputSingleTaskGP(X, Y, train_X_std=X_std)
fit_gpytorch_mll(model.make_mll())
print(model.posterior(torch.tensor([[0.25], [0.75]])).mean)


## カテゴリ確率を周辺化


In [ ]:
X_cont = torch.linspace(0, 1, 6).unsqueeze(-1)
P = torch.tensor([[0.9, 0.1], [0.8, 0.2], [0.6, 0.4], [0.4, 0.6], [0.2, 0.8], [0.1, 0.9]])
Y_cat = torch.sin(2 * torch.pi * X_cont) + 0.2 * P[:, 1:2]
cat_model = UncertainCategoricalSingleTaskGP(X_cont, P, Y_cat)
fit_gpytorch_mll(cat_model.make_mll())
post = cat_model.posterior_with_category_probabilities(
    torch.tensor([[0.5]]), torch.tensor([[0.5, 0.5]])
)
print(post.mean)


## Uncertain-input candidate proposal

連続入力の不確実性を考慮したモデルから候補を選択します。カテゴリ確率モデルは別の予測APIを使うため、確率ベクトルを固定した候補比較を行います。


In [ ]:
candidate_grid = torch.linspace(0, 1, 51).unsqueeze(-1)
model.eval()
with torch.no_grad():
    scores = model.posterior(candidate_grid).mean.squeeze(-1)
candidate = candidate_grid[scores.argmax().item():scores.argmax().item() + 1]
assert candidate.shape == (1, 1)
continuous_Y = torch.sin(2 * torch.pi * candidate)
assert torch.isfinite(continuous_Y).all()
print("uncertain-input candidate:", candidate, "objective:", continuous_Y)
cat_grid = torch.linspace(0, 1, 21).unsqueeze(-1)
cat_probs = torch.tensor([[0.5, 0.5]]).expand(len(cat_grid), -1)
cat_model.eval()
with torch.no_grad():
    cat_scores = cat_model.posterior_with_category_probabilities(
        cat_grid, cat_probs
    ).mean.squeeze(-1)
cat_candidate = cat_grid[cat_scores.argmax().item():cat_scores.argmax().item() + 1]
cat_candidate_probs = cat_probs[:1]
cat_candidate_Y = (
    torch.sin(2 * torch.pi * cat_candidate) + 0.2 * cat_candidate_probs[:, 1:2]
)
assert cat_candidate.shape == (1, 1)
assert torch.isfinite(cat_candidate_Y).all()
print("uncertain-category candidate:", cat_candidate, "objective:", cat_candidate_Y)


## 獲得関数による候補最適化とMixed型

連続入力とMixed入力の不確実性を学習データ側で表現し、候補入力は確定値として扱います。`qLogExpectedImprovement`を最適化し、Mixed型ではカテゴリを列挙します。カテゴリ確率を扱うモデルは独自Posterior APIを使うため、上記の確率固定・有限候補比較を維持します。


In [ ]:
from botorch.acquisition.logei import qLogExpectedImprovement
from botorch.optim import optimize_acqf
from botorch.optim.optimize import optimize_acqf_mixed

from robotorchan.models import MixedUncertainInputSingleTaskGP

bounds = torch.tensor([[0.0], [1.0]])
continuous_acqf = qLogExpectedImprovement(model=model, best_f=Y.max())
continuous_candidate, _ = optimize_acqf(
    acq_function=continuous_acqf,
    bounds=bounds,
    q=1,
    num_restarts=3,
    raw_samples=32,
)
print(
    "Continuous qLogEI candidate:",
    continuous_candidate,
    torch.sin(2 * torch.pi * continuous_candidate),
)

mixed_X = torch.cat(
    [X, (torch.arange(len(X)) % 2).to(X).unsqueeze(-1)],
    dim=-1,
)
mixed_Y = torch.sin(2 * torch.pi * mixed_X[:, :1]) + 0.2 * mixed_X[:, 1:2]
mixed_model = MixedUncertainInputSingleTaskGP(
    mixed_X,
    mixed_Y,
    cat_dims=[1],
    train_X_std=torch.full((len(mixed_X), 1), 0.03),
)
fit_gpytorch_mll(mixed_model.make_mll())
mixed_model.eval()
mixed_bounds = torch.tensor([[0.0, 0.0], [1.0, 1.0]])
mixed_acqf = qLogExpectedImprovement(model=mixed_model, best_f=mixed_Y.max())
mixed_candidate, _ = optimize_acqf_mixed(
    acq_function=mixed_acqf,
    bounds=mixed_bounds,
    q=1,
    num_restarts=3,
    raw_samples=32,
    fixed_features_list=[{1: 0.0}, {1: 1.0}],
)
assert mixed_candidate[0, 1].item() in (0.0, 1.0)
mixed_candidate_Y = (
    torch.sin(2 * torch.pi * mixed_candidate[:, :1])
    + 0.2 * mixed_candidate[:, 1:2]
)
print("Mixed uncertain-input candidate:", mixed_candidate, mixed_candidate_Y)
